# metrics

> Pixel-level overlap metrics (per-class IoU/Dice/precision/recall) and object-level detection metrics (connected-component hit/miss/false-alarm).

Pixel metrics tell you how well the shapes overlap; object metrics tell you how many distinct targets you actually found. In inspection a missed object is usually far costlier than a few wrong pixels, so we track both.

In [ ]:
#| default_exp metrics

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from statistics import mean
from typing import Dict, List

import cv2
import numpy as np
import torch

## Pixel metrics

We accumulate a confusion tally (TP/FP/FN) per class across an epoch, then compute overlap metrics once.

- `task='binary'` (`num_classes == 1`, default): thresholds the sigmoid for the single foreground class.
- `task='multiclass'` (`num_classes > 1`, default when `task` omitted): argmax, mean excludes background (label `0`) since classes are mutually exclusive.
- `task='multilabel'` (`num_classes > 1`, overlapping classes): each channel is thresholded independently via its own sigmoid -- no argmax, no shared background, so the mean includes every channel.

In [ ]:
# experiment: IoU of a perfect vs an off-by-one prediction
gt = torch.zeros(1, 1, 8, 8); gt[..., 2:6, 2:6] = 1
perfect = (gt > 0.5)
inter = (perfect & (gt > 0.5)).sum().item(); union = (perfect | (gt > 0.5)).sum().item()
print('perfect IoU:', inter / union)

In [ ]:
#| export
class SegMetrics:
    """Accumulate per-class pixel metrics over batches.

    Example:
        >>> m = SegMetrics(num_classes=1)
        >>> gt = torch.zeros(1, 8, 8); gt[2:6, 2:6] = 1
        >>> logits = torch.where(gt.bool(), 10.0, -10.0)[None]  # (1,1,8,8)
        >>> m.update(logits, gt[None]); round(m.compute()['iou'], 3)
        1.0
    """

    def __init__(
        self,
        num_classes: int = 1,     # 1 = binary, >1 = multi-class or multilabel
        threshold=0.5,             # float, or per-channel Sequence[float] for task='multilabel'
        task: str = None,          # 'binary' | 'multiclass' | 'multilabel'; None infers binary/multiclass from num_classes
    ):
        self.num_classes = num_classes
        self.task = task or ('binary' if num_classes == 1 else 'multiclass')
        assert self.task in ('binary', 'multiclass', 'multilabel'), f'unknown task {self.task!r}'
        if self.task == 'multilabel':
            # duck-type "sequence of per-channel thresholds" rather than isinstance(list, tuple):
            # Hydra/OmegaConf configs hand back ListConfig, not a plain list, for a YAML list.
            is_seq = hasattr(threshold, '__len__') and not isinstance(threshold, (str, bytes))
            self.threshold = list(threshold) if is_seq else [threshold] * num_classes
            self.labels = list(range(num_classes))
        else:
            self.threshold = threshold
            self.labels = [1] if num_classes == 1 else list(range(num_classes))
        self.reset()

    def reset(self) -> None:
        self.tp = {l: 0 for l in self.labels}
        self.fp = {l: 0 for l in self.labels}
        self.fn = {l: 0 for l in self.labels}

    @torch.no_grad()
    def update(self, logits: torch.Tensor, targets: torch.Tensor) -> None:
        "logits: (B, C, H, W); targets: (B, H, W) labels, or (B, C, H, W) float {0,1} for binary/multilabel."
        if self.task == 'multilabel':
            probs = torch.sigmoid(logits)  # (B, K, H, W)
            t = targets.float()
            if t.dim() == logits.dim() - 1:
                t = t.unsqueeze(1)
            for l in self.labels:
                p = probs[:, l] >= self.threshold[l]
                tt = t[:, l] >= 0.5
                self.tp[l] += int((p & tt).sum())
                self.fp[l] += int((p & ~tt).sum())
                self.fn[l] += int((~p & tt).sum())
            return
        if self.num_classes == 1:
            preds = (torch.sigmoid(logits)[:, 0] >= self.threshold).long()
            gts = targets.long().squeeze(1) if targets.dim() == 4 else targets.long()
        else:
            preds = logits.argmax(1)
            gts = targets.long()
        for l in self.labels:
            p, t = (preds == l), (gts == l)
            self.tp[l] += int((p & t).sum())
            self.fp[l] += int((p & ~t).sum())
            self.fn[l] += int((~p & t).sum())

    def compute(self) -> Dict:
        "Per-class metrics plus their mean (multi-class excludes background 0; multilabel does not)."
        eps = 1e-6
        mean_labels = self.labels if self.task == 'multilabel' else \
            [l for l in self.labels if not (self.num_classes > 1 and l == 0)]
        per, agg = {}, {'iou': [], 'dice': [], 'precision': [], 'recall': []}
        for l in self.labels:
            tp, fp, fn = self.tp[l], self.fp[l], self.fn[l]
            scores = {
                'iou': tp / (tp + fp + fn + eps),
                'dice': 2 * tp / (2 * tp + fp + fn + eps),
                'precision': tp / (tp + fp + eps),
                'recall': tp / (tp + fn + eps),
            }
            per[l] = scores
            if l in mean_labels:
                for k in agg:
                    agg[k].append(scores[k])
        out = {k: mean(v) for k, v in agg.items()}
        out['per_class'] = per
        return out

In [ ]:
# test: perfect binary prediction -> IoU 1; multi-class shape is right
gt = torch.zeros(1, 8, 8); gt[0, 2:6, 2:6] = 1
logits = torch.where(gt.bool(), 10.0, -10.0)[None]
m = SegMetrics(num_classes=1); m.update(logits, gt[None])
assert round(m.compute()['iou'], 3) == 1.0
mm = SegMetrics(num_classes=3); mm.update(torch.randn(2, 3, 8, 8), torch.randint(0, 3, (2, 8, 8)))
assert set(mm.compute()['per_class'].keys()) == {0, 1, 2}

In [ ]:
# test: multilabel SegMetrics -- independent per-channel scoring, overlap doesn't confuse it
gt3 = torch.zeros(1, 3, 8, 8)
gt3[0, 0, 2:6, 2:6] = 1   # chip
gt3[0, 1, 3:5, 3:5] = 1   # void, inside chip
gt3[0, 2, 2:6, 2:6] = 1   # lifted, same as chip here
logits3 = torch.where(gt3.bool(), 10.0, -10.0)  # perfect logits per channel
mm = SegMetrics(num_classes=3, task='multilabel')
mm.update(logits3, gt3)
out = mm.compute()
assert all(round(out['per_class'][l]['iou'], 3) == 1.0 for l in (0, 1, 2))  # each channel scored independently
assert round(out['iou'], 3) == 1.0  # mean includes all 3 channels, no background exclusion

# per-channel thresholds: a channel with its own higher threshold can still separately fail/pass
logits_mixed = gt3.clone() * 0  # all logits 0 -> sigmoid 0.5 everywhere
mm2 = SegMetrics(num_classes=3, task='multilabel', threshold=[0.5, 0.9, 0.5])
mm2.update(logits_mixed, gt3)  # 0.5 >= 0.5 -> predicts foreground everywhere for channels 0,2; not for channel 1 (0.5 < 0.9)
assert mm2.compute()['per_class'][1]['recall'] < 1.0

# regression: a real Hydra config hands back ListConfig, not list, for a YAML list --
# this broke silently once (visualize.show_pred_multilabel) until the isinstance check
# was replaced with duck typing. Lock that in here too.
from omegaconf import OmegaConf
cfg_threshold = OmegaConf.create([0.5, 0.9, 0.5])
assert not isinstance(cfg_threshold, (list, tuple))  # confirms this actually exercises the ListConfig path
mm3 = SegMetrics(num_classes=3, task='multilabel', threshold=cfg_threshold)
assert mm3.threshold == [0.5, 0.9, 0.5]

print('SegMetrics ✓  multilabel: independent per-channel IoU, per-channel thresholds, ListConfig threshold, no background exclusion')

## Object-level detection

We label connected components in the predicted and ground-truth foreground, then count a ground-truth object as *detected* if enough of it is covered, and a predicted object as a *false alarm* if it barely overlaps any target.

In [ ]:
#| export
def object_metrics(
    pred_mask: np.ndarray,        # (H, W) binary prediction
    gt_mask: np.ndarray,          # (H, W) binary ground truth
    overlap_thresh: float = 0.5,  # fraction of an object that must overlap to count
) -> Dict:
    """Connected-component detection stats for one image.

    Returns counts of total/detected/missed ground-truth objects and false alarms.
    """
    pred_bin = (pred_mask > 0).astype(np.uint8)
    gt_bin = (gt_mask > 0).astype(np.uint8)
    n_gt, gt_labels = cv2.connectedComponents(gt_bin)
    n_pred, pred_labels = cv2.connectedComponents(pred_bin)
    n_gt, n_pred = n_gt - 1, n_pred - 1
    detected = sum(
        ((gt_labels == i) & (pred_bin > 0)).sum() / ((gt_labels == i).sum() + 1e-6) >= overlap_thresh
        for i in range(1, n_gt + 1)
    )
    false_alarms = sum(
        ((pred_labels == j) & (gt_bin > 0)).sum() / ((pred_labels == j).sum() + 1e-6) < overlap_thresh
        for j in range(1, n_pred + 1)
    )
    return {'total_gt': n_gt, 'detected': int(detected), 'missed': n_gt - int(detected),
            'false_alarms': int(false_alarms),
            'detection_rate': int(detected) / (n_gt + 1e-6)}


class ObjectMetrics:
    "Accumulate `object_metrics` across many images."

    def __init__(self, overlap_thresh: float = 0.5):
        self.overlap_thresh = overlap_thresh
        self.reset()

    def reset(self) -> None:
        self.total_gt = self.detected = self.false_alarms = 0

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray) -> None:
        r = object_metrics(pred_mask, gt_mask, self.overlap_thresh)
        self.total_gt += r['total_gt']; self.detected += r['detected']; self.false_alarms += r['false_alarms']

    def compute(self) -> Dict:
        missed = self.total_gt - self.detected
        return {'total_objects': self.total_gt, 'detected': self.detected, 'missed': missed,
                'detection_rate': self.detected / (self.total_gt + 1e-6),
                'false_alarms': self.false_alarms}

In [ ]:
# test: two separate gt blobs, both predicted -> detection_rate 1.0, no false alarms
gt = np.zeros((40, 40), np.uint8); gt[5:15, 5:15] = 1; gt[25:35, 25:35] = 1
r = object_metrics(gt, gt)
assert r['total_gt'] == 2 and r['detected'] == 2 and r['false_alarms'] == 0
acc = ObjectMetrics(); acc.update(gt, gt)
assert acc.compute()['detection_rate'] == r['detection_rate']

## Multilabel hierarchy diagnostics

Multilabel channels can have an intended hierarchy that isn't enforced anywhere yet (e.g. `void` and `lifted` should stay inside `chip` -- see `docs/MULTILABEL_OVERLAP.md`). This is a read-only *measurement*, not enforcement (that's Phase 4's `apply_hierarchy`): how much of a "sub" channel's predicted area falls outside its "super" channel, per pair. Useful during training to see whether the hierarchy is emerging on its own before deciding whether Phase 4's post-process is actually needed.

In [ ]:
#| export
def subset_violation_rate(
    sub_mask: np.ndarray,    # (H, W) binary mask of the channel that should be a subset (e.g. void)
    super_mask: np.ndarray,  # (H, W) binary mask of the containing channel (e.g. chip)
) -> float:
    """Fraction of `sub_mask`'s positive pixels that fall *outside* `super_mask`.

    `0.0` means the hierarchy holds perfectly (every sub pixel is inside super);
    `1.0` means no overlap at all. Pure measurement -- does not modify either mask.
    """
    sub = sub_mask > 0
    n_sub = int(sub.sum())
    if n_sub == 0:
        return 0.0
    outside = int((sub & ~(super_mask > 0)).sum())
    return outside / n_sub


def hierarchy_diagnostics(
    masks: np.ndarray,                    # (K, H, W) binary, one channel per class
    pairs: List,                          # [(sub_idx, super_idx), ...], e.g. [(1, 0), (2, 0)] for (void,chip),(lifted,chip)
) -> Dict:
    "Run `subset_violation_rate` for each (sub, super) pair. Keys are 'sub_idx->super_idx'."
    return {f'{s}->{p}': subset_violation_rate(masks[s], masks[p]) for s, p in pairs}

In [ ]:
# test: perfect hierarchy -> 0 violation; a void pixel poking outside chip -> nonzero
chip = np.zeros((20, 20), np.uint8); chip[5:15, 5:15] = 1
void = np.zeros((20, 20), np.uint8); void[8:12, 8:12] = 1     # fully inside chip
lifted = np.zeros((20, 20), np.uint8); lifted[5:15, 5:15] = 1  # identical to chip
assert subset_violation_rate(void, chip) == 0.0
assert subset_violation_rate(lifted, chip) == 0.0

void_leaky = void.copy(); void_leaky[0:2, 0:2] = 1  # 4 pixels poke outside chip entirely
rate = subset_violation_rate(void_leaky, chip)
assert 0.0 < rate < 1.0

masks = np.stack([chip, void, lifted])  # (3, H, W): chip=0, void=1, lifted=2
diag = hierarchy_diagnostics(masks, pairs=[(1, 0), (2, 0)])
assert diag == {'1->0': 0.0, '2->0': 0.0}
print('hierarchy diagnostics ✓  0 violation for a correct hierarchy, nonzero when sub leaks outside super')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()